

# How does a bike-share navigate speedy success?
<span style="font-size: 25px;">Google case study 1</span>

# Introduction
<p style="text-align:justify;font-size:16px;margin-bottom:15px;">This is the data analytics project from Google Data Analytics Professional Certificate course. Here, I will use data analytics tools for the problem solving of the fictional company-Cyclistic. In this report, I included following deliverables:</p>
<p style="text-align:justify;font-size:16px;margin-bottom:15px;">1. Ask - A clear statement of the business task</p>
<p style="text-align:justify;font-size:16px;margin-bottom:15px;">2. Prepare - A description of all data sources used</p>
<p style="text-align:justify;font-size:16px;margin-bottom:15px;">3. Process - Documentation of any cleaning or manipulation of data</p>
<p style="text-align:justify;font-size:16px;margin-bottom:15px;">4. Analyze - A summary of your analysis</p>
<p style="text-align:justify;font-size:16px;margin-bottom:15px;">5. Share - Supporting visualizations and key findings</p>
<p style="text-align:justify;font-size:16px;margin-bottom:15px;">6. Act - Your top three recommendations based on your analysis</p>

# Ask
<span style=";font-size:16px;">According to the Scenario introduction, I should first analyze the given data to understand **how casual riders and annual members use Cyclistic bikes differently**, then work with the team to **design a new marketing strategy to convert casual riders into annual members**.</span>

# Prepare
<span style="font-size:16px;">The datasets I used for the analyze is Cyclistic’s historical trip data from **202503 to 202602**, totally 12 months. Here is the public data (https://divvy-tripdata.s3.amazonaws.com/index.html). The datasets for 12 months all have consistent 13 columns:</span><br>
<span style="font-size:16px;">**ride_id** (record ride id),</span><br>
<span style="font-size:16px;">**rideable_type** (rideable type contains electric_bike and classic_bike),</span><br>
<span style="font-size:16px;">**started_at** (record the start datetime of the ride),</span><br>
<span style="font-size:16px;">**ended_at** (record the end datetime of the ride),</span><br>
<span style="font-size:16px;">**start_station** (record the start station of the ride),</span><br>
<span style="font-size:16px;">**start_station_id** (record the station id of the start station),</span><br>
<span style="font-size:16px;">**start_lng** (record the end station of the ride),</span><br>
<span style="font-size:16px;">**start_lng** (record the station id of the end station),</span><br>
<span style="font-size:16px;">**start_lng** (record the latitude of the start station),</span><br>
<span style="font-size:16px;">**start_lng** (record the longitude of the start station),</span><br>
<span style="font-size:16px;">**member_casual** (record the latitude of the end station),</span><br>
<span style="font-size:16px;">**member_casual** (record the longitude of the end station),</span><br>
<span style="font-size:16px;">**member_casual** (record the member type, contains member and casual).</span><br>

# Process
<span style="font-size:16px;">I first downloaded the trip data from 202503 to 202602, then uploaded the .csv sheets to**MySQL** due to the huge size of the datasets. After checking the column consistent of all these 12 sheets, I combined them into a single worksheet for further data process and analysis.</span>

**MySQL:**

In [ ]:
## Combine all the sheets into new table
USE `DATA202503_202602`;
CREATE TABLE `202503_202602all` AS
SELECT * FROM `202503_divvy_tripdata`
UNION ALL
select * FROM `202504_divvy_tripdata`
UNION ALL
select * FROM `202505_divvy_tripdata`
UNION ALL
select * FROM `202506_divvy_tripdata`
UNION ALL
select * FROM `202507_divvy_tripdata`
UNION ALL
select * FROM `202508_divvy_tripdata`
UNION ALL
select * FROM `202509_divvy_tripdata`
UNION ALL
select * FROM `202510_divvy_tripdata`
UNION ALL
select * FROM `202511_divvy_tripdata`
UNION ALL
select * FROM `202512_divvy_tripdata`
UNION ALL
select * FROM `202601_divvy_tripdata`
UNION ALL
select * FROM `202602_divvy_tripdata`;

**Cleaning**<br>
<span style="font-size: 16px;">After checking the combined worksheet, I found some ‘bad data’ inside. First, after checking the time difference of the started_at and ended_at time, I found some outliers where started_at is later than ended_at. Second, some started_at time is in 202502, which will bring some problems for further analysis.</span>

**MySQL**

In [ ]:
## Data Cleaning
#remove outliers(test min, max, average, mode)
#FIND SOME started_at>ended_at
SELECT
    started_at,ended_at,
    TIMESTAMPDIFF(SECOND,started_at,ended_at) AS length1,
    SEC_TO_TIME(TIMESTAMPDIFF(SECOND,started_at,ended_at)) AS length2
FROM `202503_202602all`
ORDER BY length1, length2;
#Clean
UPDATE `202503_202602all`
SET started_at = (@temp := started_at),
    started_at = ended_at,
    ended_at = @temp
WHERE started_at > ended_at;
DELETE FROM `202503_202602all`
WHERE year_mon = '202502';

**Processing**<br>
<span style="font-size: 16px;">For more details prepared for further analysis, I added 4 more columns about the ride_length (the difference of started and ended time), day_of_week (from sunday to saturday from the started time), year_mon (the year and month from the started time), time_hms (the time written in HH:MM:SS from the started time).</span>

In [ ]:
##Add columns
ALTER TABLE `202503_202602all`
    ADD COLUMN ride_length VARCHAR(10),
    ADD COLUMN day_of_week TINYINT,
    ADD COLUMN year_mon INT,
    ADD COLUMN time_hms TIME;
UPDATE `202503_202602all`
SET
    ride_length = SEC_TO_TIME(TIMESTAMPDIFF(SECOND,started_at,ended_at)),
    day_of_week = DAYOFWEEK(started_at),
    year_mon = DATE_FORMAT(started_at,'%Y%m'),
    time_hms = TIME(started_at)
WHERE started_at IS NOT NULL;

# Analyze
<span style = "font-size: 16px;">The task of this project is to know the difference of member and casual riders. I aggregated the information by member_casual to know the difference from ride numbers, rideable types, ride length (most, mean, max, min), the most used started and ended station, and start time (Year month, Time, day).</span>

**MySQL**

In [ ]:
##Data grouped description
USE `data202503_202602`;
CREATE TABLE `Membertype_Table` AS
SELECT
     member_casual AS Member_Type,
    CONCAT(COUNT(*),'(',ROUND(COUNT(*)*100/SUM(COUNT(*))OVER(),2),')') AS Total_rides,
    ROUND(SUM(CASE WHEN rideable_type = 'classic_bike' THEN 1 ELSE 0 END)*100/COUNT(*),2) AS Classic_rides_pct,
    ROUND(SUM(CASE WHEN rideable_type = 'electric_bike' THEN 1 ELSE 0 END)*100/COUNT(*),2) AS Electric_rides_pct,
       (SELECT CONCAT(MIN(year_mon),'~',MAX(year_mon))
            FROM
            (SELECT year_mon, COUNT(*) AS freq
            FROM `202503_202602all` A2
            WHERE A2.member_casual = A1.member_casual
            GROUP BY year_mon
            ORDER BY freq DESC
            LIMIT 3) AS T1
       ) AS Year_mon_common,
      (SELECT CONCAT(MIN(time_hms),'~',MAX(time_hms))
           FROM
           (SELECT time_hms,COUNT(*) AS freq
            FROM `202503_202602all` A2
            WHERE A2.member_casual = A1.member_casual
            GROUP BY time_hms
            ORDER BY freq DESC
            LIMIT 50) AS T2
      ) AS Time_hms_common,
      (SELECT
           GROUP_CONCAT(CASE day_of_week
               WHEN 1 THEN 'Sunday'
               WHEN 2 THEN 'Monday'
               WHEN 3 THEN 'Tuesday'
               WHEN 4 THEN 'Wednesday'
               WHEN 5 THEN 'Thursday'
               WHEN 6 THEN 'Friday'
               WHEN 7 THEN 'Saturday'
           END)
           FROM
           (SELECT day_of_week,COUNT(*) AS freq
            FROM `202503_202602all` A2
            WHERE A2.member_casual = A1.member_casual
            GROUP BY day_of_week
            ORDER BY freq DESC
            LIMIT 2) AS T3
      ) AS Day_of_week_common,
      (SELECT GROUP_CONCAT(start_station_name,'(',start_station_id,')')
       FROM
           (SELECT start_station_name, start_station_id, COUNT(*) AS freq
            FROM `202503_202602all` A2
            WHERE start_station_name IS NOT NULL AND A2.member_casual = A1.member_casual
            GROUP BY start_station_name,start_station_id
            ORDER BY freq DESC
            LIMIT 5) AS T4
       ) AS Start_station_common,
       (SELECT GROUP_CONCAT(end_station_name,'(',end_station_id,')')
       FROM
           (SELECT end_station_name, end_station_id, COUNT(*) AS freq
            FROM `202503_202602all` A2
            WHERE end_station_name IS NOT NULL AND A2.member_casual = A1.member_casual
            GROUP BY end_station_name,end_station_id
            ORDER BY freq DESC
            LIMIT 5) AS T5
       ) AS End_station_common,
       MAX(ride_length) AS Max_length,
       MIN(ride_length) AS Min_length,
       SEC_TO_TIME(ROUND(AVG(TIME_TO_SEC(ride_length)))) AS Avg_length,
       (SELECT CONCAT(MIN(ride_length),'~',MAX(ride_length))
        FROM
            (SELECT ride_length,COUNT(*) AS freq
             FROM `202503_202602all` A2
             WHERE A2.member_casual = A1.member_casual
             GROUP BY ride_length
             ORDER BY freq DESC
             LIMIT 50) AS T6
        ) AS ride_length_common
FROM `202503_202602all` A1
GROUP BY member_casual
ORDER BY member_casual;

##Check common details
#Month_Year(Casual:678910 Summer,Member: 34567891011 Consistent usage year-round)
SELECT
    member_casual AS Member_type,
    year_mon AS Year_mon, COUNT(*) AS Freq,
    SEC_TO_TIME(ROUND(AVG(TIME_TO_SEC(ride_length)))) AS Avg_length
FROM `202503_202602all`
GROUP BY member_casual,year_mon
ORDER BY member_casual;
#Time(Casual:11-18,Member:15-19,7-8,11-14)
SELECT
    member_casual AS Member_type,
    HOUR(time_hms) AS Time_common, COUNT(*) AS Freq,
    SEC_TO_TIME(ROUND(AVG(TIME_TO_SEC(ride_length)))) AS Avg_length
FROM `202503_202602all`
GROUP BY member_casual,Time_common
ORDER BY member_casual;
#Day_of_week(Casual:Friday-Sunday Weekend,Member:Weekday)
SELECT
    member_casual AS Member_type,
    (CASE day_of_week
        WHEN 1 THEN 'Sunday'
        WHEN 2 THEN 'Monday'
        WHEN 3 THEN 'Tuesday'
        WHEN 4 THEN 'Wednesday'
        WHEN 5 THEN 'Thursday'
        WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday'
        END) AS Day_of_week, COUNT(*) AS Freq,
        SEC_TO_TIME(ROUND(AVG(TIME_TO_SEC(ride_length)))) AS Avg_length
FROM `202503_202602all`
GROUP BY member_casual,Day_of_week
ORDER BY member_casual;
#Start_station
(SELECT
    member_casual AS Member_type,
    start_station_name, start_station_id,start_lat,start_lng,
    COUNT(*) AS freq
FROM `202503_202602all`
WHERE start_station_name IS NOT NULL AND member_casual = 'casual'
GROUP BY member_casual, start_station_name, start_station_id, start_lat, start_lng
ORDER BY freq DESC
LIMIT 10)
UNION ALL
(SELECT
    member_casual,
    start_station_name, start_station_id,start_lat,start_lng,
    COUNT(*) AS freq
FROM `202503_202602all`
WHERE start_station_name IS NOT NULL AND member_casual = 'member'
GROUP BY member_casual, start_station_name, start_station_id, start_lat, start_lng
ORDER BY freq DESC
LIMIT 10);
#End_station
(SELECT
    member_casual AS Member_type,
    end_station_name, end_station_id,end_lat,end_lng,
    COUNT(*) AS freq
FROM `202503_202602all`
WHERE end_station_name IS NOT NULL AND member_casual = 'casual'
GROUP BY member_casual, end_station_name, end_station_id, end_lat, end_lng
ORDER BY freq DESC
LIMIT 10)
UNION ALL
(SELECT
    member_casual,
    end_station_name, end_station_id,end_lat,end_lng,
    COUNT(*) AS freq
FROM `202503_202602all`
WHERE end_station_name IS NOT NULL AND member_casual = 'member'
GROUP BY member_casual, end_station_name, end_station_id, end_lat, end_lng
ORDER BY freq DESC
LIMIT 10);

<span style = "font-size: 16px;">From the result of the above query table, i got some basic insights from the data. The casual riders (35.94%) are less than member riders (64.06%).</span><br> 
<span style = "font-size: 16px;">And for the **causal** riders: Most use Electric_bikes, Most use in summer, Most use in the afternoon(11-18), Most use during the weekend, Most use for the turiost attractions, and ride_length is longer(avg is 00:22:38); the **member** riders: Most use Electric_bikes, Consistent usage year-round, Most use during the rush hour(16-19, 7-9), Most use during the weekday, and ride_length is shorter(avg is 00:12:26).</span>

# Share
<span style = "font-size: 16px;">Using the above MySQL tables, I used Tableau for visualizations analysis.</span>

<span style = "font-size: 16px;">**Member types**: The member riders are more than casual riders, and all riders use electric bikes more than classic ones.</span>
![Member_types.jpg](attachment:0d0efd68-c1bf-41df-8642-7601070faba4.jpg)

<span style = "font-size: 16px;">**Difference in Months**: member riders have the consistent usage year-round and the average ride lengths for all 12 months are similar , but the casual riders most use the bike in summer, and the average ride lengths in summer is more than other months.But in general, the average ride lengths for member riders are much less than casual riders.</span>
![Month.jpg](attachment:b66fdd95-82ac-4704-9bf1-01e520435799.jpg)

<span style = "font-size: 16px;">**Difference in days**: Member riders most use the bike during the weekday, and casual riders most use during the weekend. And the average ride lengths for member riders are similar for all the days, but the casual riders ride longer during the weekends compared to weekdays.</span>![day1.jpg](attachment:955fb25f-c158-4f44-a13b-1ba58c7eefb8.jpg)![day2.jpg](attachment:5d5e3434-370e-48c7-9c8b-0c5acc0712b3.jpg)

<span style = "font-size: 16px;"> **Difference in hours**:Member riders most use in the morning (7-9) and in the afternoon (16-19), which is the rush hour for workers, and casual riders most use in the afternoon. </span>![hour.jpg](attachment:dc1d2366-1857-41e4-91b6-4dbf7052eddc.jpg)

<span style = "font-size: 16px;">**Difference in locations**: From the map, compared to the member riders, casual riders most use near the tourist attractions (shore area), the top two station names are Navy Pier,DuSable Lake Shore Dr & Monroe St.</span>![locations.jpg](attachment:4e13b114-133e-44bb-bae4-de7daa5a299c.jpg)

# Act
<span style = "font-size: 16px;">**Difference of member and casual riders**: According to the above analysis and data visualization, here i made three main conclusions</span><br>
<span style = "font-size: 16px;">1. Casual riders are most for holidays or weekend (most use in summer, afternoon, during the weekend, tourist attractions), and member riders are most for daily commuting to work (consistent use all year round, during rush hour and weekdays).</span><br>
<span style = "font-size: 16px;">2. The average ride length of casual riders is much longer than member riders, and casual riders also have a large number, which shows the conversion potential of the casual riders. </span><br>
<span style = "font-size: 16px;">3. There have some overlaps between casual and member riders, which can be used as the target groups for the member conversion. </span><br>
<span style = "font-size: 16px;">**Recommendations**: Based on the difference of member and casual riders, here i made three key recommendations</span><br>
<span style = "font-size: 16px;">1. Promote membership campaigns at the shore area in summer afternoon and offer discount vouchers to attract more casual riders. The campaign can also have the limited free trials near the shore area.</span><br>
<span style = "font-size: 16px;">2. Offer casual riders online discounts for the weekdays and rush hours to convert the overlapped riders.</span><br>
<span style = "font-size: 16px;">3. Promote the reward program online that can change ride length into different vouchers for member registeration.</span><br>